# IGRPO — Cooperative MARL experiments (Colab)

Runs the **Cluttered + FourRooms** × **{IPPO, MAPPO, GRPO, GRPO-global}** × **3 seeds** matrix.
Each run logs **reward**, **policy entropy**, and **peak VRAM** (`system/peak_vram_gb`) to Weights & Biases.

### Before you run
1. **Runtime → Change runtime type → GPU** (a T4 is enough). Training requires CUDA — `num_gpus_per_learner=1` is hardcoded.
2. **Push your local changes to GitHub first.** This notebook clones `ko120/IGRPO`; the FourRooms env, the updated `run_experiments.sh`, the VRAM logging, and `--n_episodes` must be on the remote. The preflight cell (step 4) fails loudly if they are missing.
3. Have your **W&B API key** ready (or use the smoke test, which runs with `--debug` and no logging).

Run the cells top to bottom.

In [ ]:
# 1) Confirm a GPU is attached (training needs CUDA).
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > GPU, then rerun."
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Clone the repo (must already contain your pushed changes).
import os
if not os.path.isdir("/content/IGRPO"):
    !git clone --depth 1 https://github.com/ko120/IGRPO.git /content/IGRPO
%cd /content/IGRPO
!git log --oneline -1

In [ ]:
# 3) Install the RL/env/logging stack pinned to the working environment.
#    torch and numpy are left as Colab's GPU defaults on purpose (do NOT override torch).
!pip install -q \
  "ray[rllib]==2.55.1" \
  "gymnasium==1.2.2" \
  "gym==0.23.1" \
  "gym-minigrid==1.0.3" \
  "wandb==0.27.0" \
  "moviepy==2.2.1" \
  "seaborn==0.13.2" \
  "pyyaml==6.0.3"
print("\nIf you see dependency/import errors below, use Runtime > Restart session, "
      "then re-run from this cell down (a one-time numpy/torch reconciliation).")

In [ ]:
# 4) Preflight: verifies the envs build, the 24-run matrix expands correctly, and
#    VRAM logging is wired in. Must end with "ALL PASS" (else your changes aren't on GitHub).
!python test_fourrooms_pipeline.py

In [ ]:
# 5) Log in to Weights & Biases (paste your API key). Skip if you only run the smoke test.
import wandb
wandb.login()

### (Optional) Persist checkpoints to Google Drive
Colab's local disk is wiped on disconnect. Run the next cell to symlink the repo's
`checkpoints/` into Drive so runs survive. `main.py` resolves `--save_path` **relative to
the repo root** (it strips a leading `/`), so the runner below uses repo-relative paths and
this symlink transparently redirects them to Drive.

In [ ]:
# 6) (Optional) Mount Drive and redirect repo/checkpoints -> Drive.
import os
from google.colab import drive
drive.mount("/content/drive")
drive_ckpt = "/content/drive/MyDrive/IGRPO_checkpoints"
os.makedirs(drive_ckpt, exist_ok=True)
repo_ckpt = "/content/IGRPO/checkpoints"
if not os.path.exists(repo_ckpt):
    os.symlink(drive_ckpt, repo_ckpt)
print("checkpoints persist to:", os.path.realpath(repo_ckpt))

### Smoke test
One short run (50 episodes, `--debug` so no W&B) to confirm the whole stack trains on this
Colab GPU and logs VRAM. Takes a few minutes (most of it is Ray/Torch warmup). Look for a
printed checkpoint path and rising "Mean reward" lines.

In [ ]:
# 7) SMOKE TEST — end-to-end sanity on one config.
!python main.py --algorithm IPPO --mode grpo --use_return \
    --seed 42 --env_name MultiGrid-FourRooms-15x15-v0 \
    --n_episodes 50 --debug

### Full experiment matrix
Runs **2 envs × 4 configs × 3 seeds = 24 runs**, **sequentially** (one at a time) — the simplest,
most robust path on a single Colab GPU: one Ray cluster at a time, no VRAM pile-up, no Ray
startup storm.

**Reality check:** the paper setting is `N_EPISODES = 30000` per run; 24 of those will **not**
finish in a free Colab session. Start with a small `N_EPISODES` (e.g. 2000) and/or trim
`ENVS`/`SEEDS`. Each run streams to W&B, so partial progress is never lost.

In [ ]:
# 8) Sequential matrix runner. Mirrors run_experiments.sh but one run at a time.
import itertools, subprocess

ENVS = ["MultiGrid-Cluttered-Fixed-15x15", "MultiGrid-FourRooms-15x15-v0"]
SEEDS = [42, 1, 7]
CONFIGS = {
    "IPPO":                    ["--algorithm", "IPPO",  "--mode", "ppo"],
    "MAPPO":                   ["--algorithm", "MAPPO", "--mode", "ppo"],
    "grpo_return":             ["--algorithm", "IPPO",  "--mode", "grpo", "--use_return"],
    "grpo_global_norm_return": ["--algorithm", "IPPO",  "--mode", "grpo", "--global_norm", "--use_return"],
}
N_EPISODES = 2000           # paper uses 30000; keep small for Colab, raise if you have time
WB_PROJECT = "multigrid-ippo"

for env, (cname, cflags) in itertools.product(ENVS, CONFIGS.items()):
    for seed in SEEDS:
        # repo-relative path (main.py strips a leading "/"); Drive symlink persists it.
        save = f"checkpoints/{env}/{cname}/seed_{seed}"
        cmd = ["python", "main.py", *cflags, "--seed", str(seed),
               "--env_name", env, "--wandb_project", WB_PROJECT,
               "--save_path", save, "--n_episodes", str(N_EPISODES)]
        print("\n>>>", " ".join(cmd), flush=True)
        subprocess.run(cmd, check=False)
print("\nMatrix complete.")

### Alternative: the repo's parallel script
`run_experiments.sh` runs the matrix with a **concurrency cap** (`MAX_PARALLEL`, default 3) and
**staggered Ray startups**, so it avoids the *"raylet failed to startup / GCS overloaded"* crash
and the VRAM blow-up you'd get launching all 24 at once. Tune it for your machine, e.g.
`MAX_PARALLEL=2 bash run_experiments.sh`. The sequential runner above is still simplest on one GPU.

In [ ]:
# !bash run_experiments.sh

In [ ]:
# 9) Inspect what was produced.
!find checkpoints -maxdepth 4 -name "*.json" -o -maxdepth 4 -type d 2>/dev/null | sort | head -60